# SkillGraph Week 2 — Module 4: Multi-Tier Skill Normalization & Taxonomy Resolution

**Project:** SkillGraph — An Explainable Knowledge Graph-Based Framework for Career-Oriented Skill Gap Analysis and Learning Path Recommendation  
**Notebook:** `05_skill_normalization.ipynb`  
**Module:** `SkillNormalizer` (`models/skill_extraction/dictionary_extractor.py`)

---

## 1. Research Objective & Normalization Ethics

Mapping surface mentions (`"Postgres"`, `"K8s"`, `"ml"`, `"PDPA compliance"`, `"IFRS"`) to canonical taxonomy identifiers (`SG-SK-0001`..`SG-SK-0230`) requires strict rules to prevent silent semantic corruption:
1. **Tier 1 (`exact_match`, confidence 1.00, `MAPPED`):** Direct canonical match (e.g., `"Python"`, `"SIEM"`, `"SQL"`, `"Power BI"`).
2. **Tier 2 (`alias_match`, confidence 0.95, `MAPPED`):** Verified synonym, acronym, or safe orthographic variant from taxonomy metadata (e.g., `"Postgres"` $\to$ `PostgreSQL`, `"nlp"` $\to$ `Natural Language Processing`, `"sklearn"` $\to$ `scikit-learn`).
3. **Tier 3 (`context_verified_short_token`, confidence 0.90, `MAPPED`):** Ambiguous 1–2 letter programming tokens (`C`, `R`, `Go`) accepted **only** with strict casing and non-idiomatic local context (rejecting `"Vitamin C"`, `"C-suite"`, `"R&D"`, `"ready to go"`, and Title-case `"Siem"` in `"Siem Reap"`).
4. **Tier 4 (`country_specific_broad_match`, confidence 0.80, `BROAD_COUNTRY_MAPPING` vs. `UNRESOLVED_CANONICAL`):** Under `data/annotations/annotation_guidelines.md` (*"Different laws, standards or tools are different skills even if they play a similar role"*), national statutes (`PDPA compliance`, `GDPR compliance`) and local accounting frameworks (`IFRS`, `K-IFRS`, `US GAAP`) remain `UNRESOLVED_CANONICAL` (`skill_id = None`) in strict mode unless broad mapping to `SG-SK-0048` / `SG-SK-0058` is explicitly requested.
5. **Zero Taxonomy Fabrication:** Unknown mentions (`"TensorRT-XYZ"`) always return `UNRESOLVED_CANONICAL` with `skill_id = None`.

In [1]:
import json
import sys
from pathlib import Path
from collections import Counter

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.skill_extraction import SkillNormalizer, load_taxonomy_dataframe

DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATA_ANNOTATIONS_DIR = PROJECT_ROOT / 'data' / 'annotations'
OUTPUTS_EXPORTS_DIR = PROJECT_ROOT / 'outputs' / 'exports'

df_tax_230 = load_taxonomy_dataframe(DATA_PROCESSED_DIR / 'global_skill_taxonomy.csv')
df_tax_31 = load_taxonomy_dataframe(OUTPUTS_EXPORTS_DIR / 'skill_taxonomy.json')

print(f'Global Taxonomy (CSV): {len(df_tax_230)} canonical skills, {df_tax_230["skill_category"].nunique()} categories, {df_tax_230["skill_type"].nunique()} skill types')
print(f'Benchmark Taxonomy (JSON): {len(df_tax_31)} canonical skills, {df_tax_31["skill_category"].nunique()} categories')
display(df_tax_230.groupby(['skill_type', 'skill_category']).size().reset_index(name='skill_count').head(12))

Global Taxonomy (CSV): 230 canonical skills, 18 categories, 8 skill types
Benchmark Taxonomy (JSON): 31 canonical skills, 11 categories


,skill_type,skill_category,skill_count
0,Certification,Business and finance,1
1,Certification,Cybersecurity,1
2,Certification,Healthcare competencies,1
3,Certification,Industry-specific skills,3
4,Certification,Networking,1
5,Certification,Project management,2
6,Competency,Communication,1
7,Competency,Healthcare competencies,1
8,Competency,Industry-specific skills,1
9,Hard skill,Business and finance,9


## 2. Normalization Resolution Probes (Strict vs. Broad & Context Disambiguation)

We test `SkillNormalizer` on canonical names, abbreviations (`nlp`, `ml`, `sql`), technical punctuation (`C++`, `Node.js`, `CI/CD`), false-positive traps (`Siem`, `Vitamin C`, `R&D`, `ready to go`), regional privacy/accounting standards, and out-of-vocabulary strings.

In [2]:
norm_strict_230 = SkillNormalizer(DATA_PROCESSED_DIR / 'global_skill_taxonomy.csv', include_broad_country_mappings=False)
norm_broad_230 = SkillNormalizer(DATA_PROCESSED_DIR / 'global_skill_taxonomy.csv', include_broad_country_mappings=True)
norm_tech_31 = SkillNormalizer(OUTPUTS_EXPORTS_DIR / 'skill_taxonomy.json', include_broad_country_mappings=False)

probes = [
    ('Python', norm_strict_230, None, False),
    ('power bi', norm_strict_230, None, False),
    ('Postgres', norm_strict_230, None, False),
    ('nlp', norm_strict_230, None, False),
    ('ml', norm_strict_230, None, False),
    ('sql', norm_strict_230, None, False),
    ('SIEM', norm_strict_230, None, False),
    ('Siem', norm_strict_230, 'Based in Siem Reap, Cambodia', False),
    ('PDPA compliance', norm_strict_230, None, False),
    ('PDPA compliance', norm_broad_230, None, True),
    ('IFRS', norm_broad_230, None, True),
    ('UnknownProprietaryTool', norm_strict_230, None, False),
    ('C++', norm_tech_31, 'Systems engineering in C++ and C', False),
    ('Node.js', norm_tech_31, None, False),
    ('Go', norm_tech_31, 'Microservices written in Go and Python', False),
    ('Go', norm_tech_31, 'Candidates ready to go to production', False),
    ('C', norm_tech_31, 'Clinical trial on Vitamin C supplementation', False),
    ('R', norm_tech_31, 'Partnering with the medical R&D lab', False),
]

rows = []
for mention, normalizer, ctx, _ in probes:
    res = normalizer.normalize(mention, context_text=ctx)
    rows.append({
        'Input Mention': mention,
        'Context': ctx or '(standalone)',
        'Skill ID': res['skill_id'],
        'Canonical Skill': res['canonical_skill'],
        'Match Method': res['match_method'],
        'Status': res['mapping_status'],
        'Confidence': res['mapping_confidence'],
    })
display(pd.DataFrame(rows))

,Input Mention,Context,Skill ID,Canonical Skill,Match Method,Status,Confidence
0,Python,(standalone),SG-SK-0001,Python,exact_match,MAPPED,1.00
1,power bi,(standalone),SG-SK-0022,Power BI,exact_match,MAPPED,1.00
2,Postgres,(standalone),SG-SK-0015,PostgreSQL,alias_match,MAPPED,0.95
3,nlp,(standalone),SG-SK-0031,Natural Language Processing,alias_match,MAPPED,0.95
4,ml,(standalone),SG-SK-0027,Machine Learning,alias_match,MAPPED,0.95
5,sql,(standalone),SG-SK-0006,SQL,exact_match,MAPPED,1.00
6,SIEM,(standalone),SG-SK-0044,SIEM,exact_match,MAPPED,1.00
7,Siem,"Based in Siem Reap, Cambodia",NaN,NaN,unresolved_canonical,UNRESOLVED_CANONICAL,0.00
8,PDPA compliance,(standalone),NaN,NaN,unresolved_canonical,UNRESOLVED_CANONICAL,0.00
9,PDPA compliance,(standalone),SG-SK-0048,Data Privacy Compliance,country_specific_broad_match,BROAD_COUNTRY_MAPPING,0.80


## 3. Audit of All 21,714 Corpus Mentions & 530 Unconfirmed Candidate Spans

We audit how `SkillNormalizer` classifies every mention recorded in `data/processed/global_skill_mentions.csv` (21,714 spans) and every provisional candidate span in `data/annotations/human_annotation_sample_50.json` (530 spans, **unconfirmed human review pending**).

In [3]:
df_mentions = pd.read_csv(DATA_PROCESSED_DIR / 'global_skill_mentions.csv')
strict_statuses = Counter()
broad_statuses = Counter()
unresolved_examples = Counter()

for mention in df_mentions['skill_mention']:
    r_strict = norm_strict_230.normalize(mention)
    r_broad = norm_broad_230.normalize(mention)
    strict_statuses[r_strict['match_method']] += 1
    broad_statuses[r_broad['match_method']] += 1
    if r_strict['mapping_status'] == 'UNRESOLVED_CANONICAL':
        unresolved_examples[mention] += 1

print('=== 21,714 Corpus Mentions Normalization Breakdown ===')
print('Strict Mode (default):', dict(strict_statuses))
print('Broad Country Mode:   ', dict(broad_statuses))
print(f'Distinct Regional/Unresolved Surface Strings in Strict Mode: {len(unresolved_examples)} (total occurrences: {sum(unresolved_examples.values())})')
display(pd.DataFrame(unresolved_examples.most_common(12), columns=['Surface Form', 'Count']))

with open(DATA_ANNOTATIONS_DIR / 'human_annotation_sample_50.json', 'r', encoding='utf-8') as f:
    sample_50 = json.load(f)

cand_counts = Counter()
cand_unresolved = []
for rec in sample_50['records']:
    for sp in rec['pass_1']['spans']:
        r_s = norm_strict_230.normalize(sp['skill_mention'])
        cand_counts[r_s['mapping_status']] += 1
        if r_s['mapping_status'] == 'UNRESOLVED_CANONICAL':
            cand_unresolved.append((rec['job_id'], sp['span_id'], sp['skill_mention']))

print('\n=== 50-Record Unconfirmed Candidate Pool (530 Spans) Strict Audit ===')
print('Strict Normalization Status Counts:', dict(cand_counts))
print('Regional/Broad Privacy Phrases Held Unresolved (including 4 named statutory laws):', cand_unresolved)

=== 21,714 Corpus Mentions Normalization Breakdown ===
Strict Mode (default): {'alias_match': 13856, 'exact_match': 7707, 'unresolved_canonical': 151}
Broad Country Mode:    {'alias_match': 13856, 'exact_match': 7707, 'country_specific_broad_match': 151}
Distinct Regional/Unresolved Surface Strings in Strict Mode: 35 (total occurrences: 151)


,Surface Form,Count
0,PDPA compliance,19
1,Vietnamese personal data protection rules,9
2,POPIA compliance,8
3,data protection requirements,8
4,LGPD compliance,7
5,IFRS,7
6,GDPR compliance,7
7,PIPEDA compliance,6
8,UAE PDPL compliance,6
9,Australian Privacy Act compliance,6



=== 50-Record Unconfirmed Candidate Pool (530 Spans) Strict Audit ===
Strict Normalization Status Counts: {'MAPPED': 525, 'UNRESOLVED_CANONICAL': 5}
Regional/Broad Privacy Phrases Held Unresolved (including 4 named statutory laws): [('GLOBAL_SYN_001733', 'R09-S15', 'GDPR compliance'), ('GLOBAL_SYN_001400', 'R14-S09', 'PIPEDA compliance'), ('GLOBAL_SYN_000098', 'R18-S09', 'PDPA compliance'), ('GLOBAL_SYN_000951', 'R19-S08', 'DPDP Act compliance'), ('GLOBAL_SYN_001784', 'R21-S12', 'data protection requirements')]
